# Phase 3: Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis on the raw customer churn dataset.
No preprocessing, encoding, imputing, or modeling is performed - this is analysis only.

In [ ]:
import sys
sys.path.append('..')

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_raw_data

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load raw data
df = load_raw_data(path="../data/raw/telco_customer_churn.csv")
df.head()

## Target Distribution

In [ ]:
# Bar plot of Churn value counts
churn_counts = df['Churn'].value_counts()
plt.figure(figsize=(8, 5))
sns.barplot(x=churn_counts.index, y=churn_counts.values, palette='viridis')
plt.title('Distribution of Churn (Target Variable)')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.show()

# Print churn rate as percentage
churn_rate = (df['Churn'] == 'Yes').mean() * 100
print(f"Overall Churn Rate: {churn_rate:.2f}%")

## Numeric Feature Distributions

In [ ]:
# Histograms for numeric features
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Convert TotalCharges to numeric for visualization only (using errors='coerce')
df_viz = df.copy()
df_viz['TotalCharges'] = pd.to_numeric(df_viz['TotalCharges'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(numeric_cols):
    sns.histplot(df_viz[col].dropna(), kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Numeric Features vs Target

In [ ]:
# Boxplots of numeric features grouped by Churn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df_viz, x='Churn', y=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} by Churn Status')
    axes[i].set_xlabel('Churn')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

## Categorical Features vs Target

In [ ]:
# Count plots showing Churn rate by categorical features
categorical_features = ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for i, feature in enumerate(categorical_features):
    # Create grouped bar chart
    sns.countplot(data=df, x=feature, hue='Churn', ax=axes[i], palette='Set2')
    axes[i].set_title(f'Churn Distribution by {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(title='Churn')

plt.tight_layout()
plt.show()

In [ ]:
# Print churn rate tables for each categorical feature
for feature in categorical_features:
    print(f"\n{'='*50}")
    print(f"Churn Rate by {feature}")
    print(f"{'='*50}")
    
    churn_table = df.groupby(feature)['Churn'].apply(
        lambda x: (x == 'Yes').mean() * 100
    ).round(2)
    
    for category, rate in churn_table.items():
        print(f"{category}: {rate}%")

## Correlation Heatmap

In [ ]:
# Correlation heatmap for numeric columns only
numeric_cols_corr = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
df_corr = df_viz[numeric_cols_corr].copy()

# Calculate correlation matrix
correlation_matrix = df_corr.corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

## Observations

Based on the exploratory data analysis above, here are the key observations:

1. **Contract Type Impact**: Customers with month-to-month contracts have a significantly higher churn rate compared to those with one-year or two-year contracts. This suggests that contract length is a strong predictor of customer retention.

2. **Tenure Relationship**: Customers with shorter tenure (newer customers) are more likely to churn, as shown in the boxplot where churned customers tend to have lower tenure values. This indicates that customer loyalty builds over time.

3. **Internet Service Effect**: Customers with fiber optic internet service show higher churn rates compared to those with DSL or no internet service. This may indicate service quality or pricing issues specific to fiber optic offerings.

4. **Monthly Charges Pattern**: Churned customers tend to have higher monthly charges on average compared to non-churned customers, suggesting that price sensitivity plays a role in churn decisions.

5. **Correlation Insights**: There is a strong positive correlation (0.83) between tenure and TotalCharges, which is expected as longer-tenured customers accumulate more total charges over time. MonthlyCharges also shows moderate positive correlation with TotalCharges.

**Note**: Correlation does not imply causation. The observed relationships and patterns indicate associations between variables, but do not prove that one variable causes changes in another.